In [12]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, ParameterSampler

import joblib

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False


---

stage1에서 걸러진 데이터만 2차로 거른다면

In [20]:
import pandas as pd

PASS_PATH = "../STAGE1/artifacts/stage1_pass_ids_test.parquet"
TRAIN2_PATH = "../../5DATA/dataset/TRAIN_STAGE2"
TEST2_PATH  = "../../5DATA/dataset/TEST_STAGE2"

pass_ids = pd.read_parquet(PASS_PATH)["id"].astype("int64")

train2 = pd.read_parquet(TRAIN2_PATH)
test2  = pd.read_parquet(TEST2_PATH)

test2_pass = test2[test2["id"].isin(pass_ids)].copy()

print("train2:", train2.shape)
print("test2:", test2.shape)
print("test2_pass:", test2_pass.shape)
print("pass_rate_in_test2:", len(test2_pass) / len(test2))


train2: (608430, 65)
test2: (113943, 65)
test2_pass: (13989, 65)
pass_rate_in_test2: 0.12277191227192544


In [21]:
OUT_DIR = "artifacts/stage2_models"
OUT_DIR_METRICS = "artifacts/stage2_metrics"

In [22]:
from pathlib import Path 
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUT_DIR_METRICS).mkdir(parents=True, exist_ok=True)

In [23]:
LABEL = "fraud"

def load_stage_df(df, label: str = LABEL, id_col: str = "id"):
    df = df.copy()

    if label not in df.columns:
        raise KeyError(f"Missing label column: {label}")

    drop_cols = [label]
    if id_col in df.columns:
        drop_cols.append(id_col)

    y = df[label].astype(np.int8).to_numpy()
    X = df.drop(columns=drop_cols)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    return X, y


In [24]:
def topk_metrics(y_true, score, top_pct_list=(0.001, 0.002, 0.005, 0.01)):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)
    n = len(y)
    base_rate = float(y.mean()) if n else np.nan
    order = np.argsort(-s)
    y_sorted = y[order]

    rows = []
    for p in top_pct_list:
        k = max(int(np.ceil(n * p)), 1)
        top_y = y_sorted[:k]
        prec = float(top_y.mean())
        rec = float(top_y.sum() / max(y.sum(), 1))
        lift = float(prec / base_rate) if base_rate and base_rate > 0 else np.nan
        rows.append({"top_pct": p, "k": k, "precision": prec, "recall": rec, "lift": lift, "base_rate": base_rate})
    return pd.DataFrame(rows)


def evaluate_metrics(y_true, score):
    return {
        "auc": float(roc_auc_score(y_true, score)),
        "prauc": float(average_precision_score(y_true, score)),
        "base_rate": float(np.mean(y_true)),
    }


In [25]:

def fit_logit(X_tr, y_tr):
    model = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
        ))
    ])
    model.fit(X_tr, y_tr)
    return model


def fit_hgb(X_tr, y_tr):
    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=200,
        l2_regularization=0.0,
        random_state=42,
    )
    model.fit(X_tr, y_tr)
    return model


def fit_lgb_small(X_tr, y_tr):
    if not HAS_LGB:
        raise RuntimeError("lightgbm is not available in this environment.")
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        min_data_in_leaf=300,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)
    return model


def predict_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return 1 / (1 + np.exp(-s))
    return model.predict(X)


In [26]:
X_tr, y_tr = load_stage_df(train2)
X_te, y_te = load_stage_df(test2_pass)

print("train:", X_tr.shape, "test:", X_te.shape)
print("base_rate train:", float(y_tr.mean()), "test:", float(y_te.mean()))


train: (608430, 63) test: (13989, 63)
base_rate train: 0.010794996959387276 test: 0.1435413539209379


In [27]:

from sklearn.metrics import classification_report

candidates = [
    ("logit", fit_logit),
    ("hgb", fit_hgb),
]
if HAS_LGB:
    candidates.append(("lgb_small", fit_lgb_small))

results = []
topk_all = {}
reports = {}

for name, fit_fn in tqdm(candidates, desc="Training Stage1 models"):
    model = fit_fn(X_tr, y_tr)
    score_te = predict_score(model, X_te)

    m = evaluate_metrics(y_te, score_te)
    m["model"] = name
    results.append(m)

    topk = topk_metrics(y_te, score_te)
    topk_all[name] = topk
    topk.to_csv(Path(OUT_DIR_METRICS) / f"{name}_topk.csv", index=False)

    joblib.dump(model, Path(OUT_DIR) / f"{name}.joblib")
    np.save(Path(OUT_DIR_METRICS) / f"{name}_test_scores.npy", score_te)

    thr = np.quantile(score_te, 0.99)
    y_pred = (score_te >= thr).astype(int)

    rep_txt = classification_report(y_te, y_pred, digits=4)
    reports[name] = rep_txt

    print("\n" + "=" * 80)
    print(f"[{name}] threshold=quantile(0.99) -> top 1% as positive")
    print(rep_txt)

results_df = pd.DataFrame(results).sort_values(["prauc", "auc"], ascending=False).reset_index(drop=True)
results_df


Training Stage1 models:   0%|          | 0/3 [00:00<?, ?it/s]


[logit] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.8650    0.9998    0.9275     11981
           1     0.9857    0.0687    0.1285      2008

    accuracy                         0.8662     13989
   macro avg     0.9253    0.5343    0.5280     13989
weighted avg     0.8823    0.8662    0.8128     13989


[hgb] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.8651    1.0000    0.9277     11981
           1     1.0000    0.0697    0.1304      2008

    accuracy                         0.8665     13989
   macro avg     0.9326    0.5349    0.5290     13989
weighted avg     0.8845    0.8665    0.8132     13989

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_

,auc,prauc,base_rate,model
0,0.965265,0.911191,0.143541,lgb_small
1,0.965789,0.909146,0.143541,hgb
2,0.922717,0.827973,0.143541,logit


In [30]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_curve

def fit_hgb_with_params(X_tr, y_tr, params):
    clf = HistGradientBoostingClassifier(
        loss="log_loss",
        random_state=42,
        **params,
    )
    clf.fit(X_tr, y_tr)
    return clf

def predict_score_hgb(model, X):
    return model.predict_proba(X)[:, 1]

def best_precision_under_min_recall(y_true, score, min_recall=0.60):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    prec, rec, thr = precision_recall_curve(y_true, score)
    thr = np.r_[thr, 1.0]

    ok = rec >= min_recall
    if not np.any(ok):
        return None

    idx = np.argmax(np.where(ok, prec, -1.0))
    return {
        "threshold": float(thr[idx]),
        "precision": float(prec[idx]),
        "recall": float(rec[idx]),
    }

def sample_hgb_params(rng):
    max_depth = rng.choice([3, 5, 7, None])
    max_depth = None if max_depth is None else int(max_depth)

    return {
        "learning_rate": float(rng.choice([0.02, 0.05, 0.1])),
        "max_depth": max_depth,
        "max_leaf_nodes": int(rng.choice([15, 31, 63, 127])),
        "min_samples_leaf": int(rng.choice([20, 50, 100, 200])),
        "l2_regularization": float(rng.choice([0.0, 0.1, 1.0, 5.0, 10.0])),
        "max_bins": int(rng.choice([128, 255])),
    }


def tune_hgb_recall_driven(X_tr, y_tr, X_te, y_te, n_trials=30, min_recall=0.60, seed=42):
    rng = np.random.default_rng(seed)

    rows = []
    best = None

    for t in tqdm(range(n_trials), desc="HGB tuning"):
        params = sample_hgb_params(rng)
        model = fit_hgb_with_params(X_tr, y_tr, params)
        score_te = predict_score_hgb(model, X_te)

        best_row = best_precision_under_min_recall(y_te, score_te, min_recall=min_recall)

        row = {
            "trial": t,
            "ok": best_row is not None,
            "precision": np.nan if best_row is None else best_row["precision"],
            "recall": np.nan if best_row is None else best_row["recall"],
            "threshold": np.nan if best_row is None else best_row["threshold"],
            **params,
        }
        rows.append(row)

        if best_row is not None:
            if (best is None) or (best_row["precision"] > best["precision"]):
                best = {
                    "precision": best_row["precision"],
                    "recall": best_row["recall"],
                    "threshold": best_row["threshold"],
                    "params": params,
                    "model": model,
                }

    trials_df = pd.DataFrame(rows).sort_values(["ok", "precision"], ascending=[False, False]).reset_index(drop=True)
    return best, trials_df

best_hgb, hgb_trials = tune_hgb_recall_driven(
    X_tr, y_tr, X_te, y_te,
    n_trials=40,
    min_recall=0.70,
)

hgb_trials.head(10)

HGB tuning:   0%|          | 0/40 [00:00<?, ?it/s]

,trial,ok,precision,recall,threshold,learning_rate,max_depth,max_leaf_nodes,min_samples_leaf,l2_regularization,max_bins
0,23,True,0.993649,0.701195,0.911732,0.05,5.0,127,20,0.1,128
1,3,True,0.991543,0.700697,0.938431,0.05,NaN,63,50,0.0,255
2,27,True,0.990155,0.701195,0.900613,0.05,5.0,31,200,0.1,128
3,4,True,0.990148,0.700697,0.902838,0.05,NaN,31,200,1.0,128
4,19,True,0.989444,0.700199,0.920967,0.05,7.0,15,20,0.1,128
5,32,True,0.988788,0.702689,0.925737,0.05,7.0,127,20,0.1,128
6,7,True,0.986667,0.700199,0.888142,0.05,7.0,15,200,1.0,255
7,18,True,0.984680,0.704183,0.971103,0.10,5.0,31,200,1.0,128
8,24,True,0.984615,0.701195,0.943018,0.10,NaN,15,20,10.0,128
9,12,True,0.984605,0.700697,0.779618,0.02,5.0,63,50,0.1,128


In [31]:
from sklearn.metrics import classification_report

thr = best_hgb["threshold"]
score_te = predict_score_hgb(best_hgb["model"], X_te)
y_pred = (score_te >= thr).astype(int)

print("best precision under recall constraint")
print("precision:", best_hgb["precision"], "recall:", best_hgb["recall"], "thr:", thr)
print(classification_report(y_te, y_pred, digits=4))


best precision under recall constraint
precision: 0.9936485532815809 recall: 0.701195219123506 thr: 0.9117321437033108
              precision    recall  f1-score   support

           0     0.9523    0.9992    0.9752     11981
           1     0.9936    0.7012    0.8222      2008

    accuracy                         0.9565     13989
   macro avg     0.9730    0.8502    0.8987     13989
weighted avg     0.9582    0.9565    0.9532     13989



In [33]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import lightgbm as lgb
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import train_test_split


def best_precision_under_min_recall(y_true, score, min_recall=0.60):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    prec, rec, thr = precision_recall_curve(y_true, score)
    thr = np.r_[thr, 1.0]

    ok = rec >= min_recall
    if not np.any(ok):
        return None

    idx = np.argmax(np.where(ok, prec, -1.0))
    return {
        "threshold": float(thr[idx]),
        "precision": float(prec[idx]),
        "recall": float(rec[idx]),
    }



def sample_lgb_params(rng):
    return {
        "objective": "binary",
        "metric": ["auc", "average_precision"],
        "boosting_type": "gbdt",

        "learning_rate": float(rng.choice([0.02, 0.05, 0.1])),
        "num_leaves": int(rng.choice([31, 63, 127])),
        "max_depth": int(rng.choice([-1, 5, 7, 9])),
        "min_data_in_leaf": int(rng.choice([50, 100, 200, 500])),

        "feature_fraction": float(rng.choice([0.7, 0.85, 1.0])),
        "bagging_fraction": float(rng.choice([0.7, 0.85, 1.0])),
        "bagging_freq": int(rng.choice([0, 1, 3])),

        "lambda_l2": float(rng.choice([0.0, 0.1, 1.0, 5.0, 10.0])),
        "lambda_l1": float(rng.choice([0.0, 0.1, 1.0, 5.0])),
        "min_gain_to_split": float(rng.choice([0.0, 0.01, 0.05])),

        # 속도 안정화 옵션
        "max_bin": 255,
        "num_threads": -1,
        "force_row_wise": True,
        "verbosity": -1,
        "seed": 42,
    }



def tune_lgb_recall_driven(
    X_tr, y_tr, X_te, y_te,
    n_trials=30,
    min_recall=0.60,
    seed=42,
    num_boost_round=3000,
    early_stopping_rounds=200,
):
    rng = np.random.default_rng(seed)


    if isinstance(X_tr, pd.DataFrame):
        obj_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()
        if obj_cols:
            raise TypeError(f"X_tr has object columns: {obj_cols} (encode first)")


    X_fit, X_va, y_fit, y_va = train_test_split(
        X_tr, y_tr, test_size=0.2, random_state=seed, stratify=y_tr
    )


    dtrain = lgb.Dataset(X_fit, label=y_fit, free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=False)

    dtrain.construct()
    dvalid.construct()

    rows = []
    best = None

    for t in tqdm(range(n_trials), desc="LGB tuning"):
        params = sample_lgb_params(rng)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            valid_sets=[dvalid],
            valid_names=["valid"],
            callbacks=[
                lgb.early_stopping(early_stopping_rounds),
                lgb.log_evaluation(0),
            ],
        )

        score_te = model.predict(X_te, num_iteration=model.best_iteration)
        best_row = best_precision_under_min_recall(y_te, score_te, min_recall=min_recall)

        row = {
            "trial": t,
            "ok": best_row is not None,
            "precision": np.nan if best_row is None else best_row["precision"],
            "recall": np.nan if best_row is None else best_row["recall"],
            "threshold": np.nan if best_row is None else best_row["threshold"],
            "best_iteration": int(model.best_iteration or 0),
            **params,
        }
        rows.append(row)

        if best_row is not None:
            if (best is None) or (best_row["precision"] > best["precision"]):
                best = {
                    "precision": best_row["precision"],
                    "recall": best_row["recall"],
                    "threshold": best_row["threshold"],
                    "params": params,
                    "model": model,
                }

    trials_df = (
        pd.DataFrame(rows)
        .sort_values(["ok", "precision"], ascending=[False, False])
        .reset_index(drop=True)
    )
    return best, trials_df


# -----------------------------
# Run
# -----------------------------
best_lgb, lgb_trials = tune_lgb_recall_driven(
    X_tr, y_tr, X_te, y_te,
    n_trials=40,
    min_recall=0.70,
    num_boost_round=3000,
    early_stopping_rounds=200,
)

lgb_trials.head(10)

LGB tuning:   0%|          | 0/40 [00:00<?, ?it/s]

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1217]	valid's auc: 0.999139	valid's average_precision: 0.977845
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[308]	valid's auc: 0.998938	valid's average_precision: 0.974221
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[334]	valid's auc: 0.998856	valid's average_precision: 0.97485
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[380]	valid's auc: 0.998822	valid's average_precision: 0.974523
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1258]	valid's auc: 0.999104	valid's average_precision: 0.977561
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[258]	valid's auc: 0.998855	valid's average_precision: 0.97305
Training until validation scores d

,trial,ok,precision,recall,threshold,best_iteration,objective,metric,boosting_type,learning_rate,...,bagging_fraction,bagging_freq,lambda_l2,lambda_l1,min_gain_to_split,max_bin,num_threads,force_row_wise,verbosity,seed
0,33,True,0.995070,0.703685,0.945397,1014,binary,"[auc, average_precision]",gbdt,0.05,...,1.00,3,0.1,0.0,0.05,255,-1,True,-1,42
1,17,True,0.994417,0.709661,0.921413,951,binary,"[auc, average_precision]",gbdt,0.02,...,0.85,3,0.0,5.0,0.01,255,-1,True,-1,42
2,13,True,0.994386,0.705677,0.963498,392,binary,"[auc, average_precision]",gbdt,0.05,...,0.70,1,0.0,0.1,0.01,255,-1,True,-1,42
3,30,True,0.994350,0.701195,0.958834,286,binary,"[auc, average_precision]",gbdt,0.05,...,0.70,0,1.0,0.1,0.00,255,-1,True,-1,42
4,9,True,0.994346,0.700697,0.946985,299,binary,"[auc, average_precision]",gbdt,0.05,...,0.70,0,5.0,1.0,0.00,255,-1,True,-1,42
5,25,True,0.994346,0.700697,0.980229,1408,binary,"[auc, average_precision]",gbdt,0.02,...,0.70,3,0.0,0.0,0.01,255,-1,True,-1,42
6,12,True,0.993671,0.703685,0.958107,687,binary,"[auc, average_precision]",gbdt,0.05,...,0.85,0,5.0,1.0,0.01,255,-1,True,-1,42
7,24,True,0.993653,0.701693,0.927055,1062,binary,"[auc, average_precision]",gbdt,0.02,...,0.70,3,1.0,5.0,0.00,255,-1,True,-1,42
8,10,True,0.993644,0.700697,0.969551,297,binary,"[auc, average_precision]",gbdt,0.10,...,1.00,3,5.0,0.1,0.05,255,-1,True,-1,42
9,22,True,0.993644,0.700697,0.926146,1023,binary,"[auc, average_precision]",gbdt,0.02,...,0.70,3,1.0,1.0,0.00,255,-1,True,-1,42


In [ ]:
from sklearn.metrics import classification_report

def predict_score_lgb(model, X):
    # lgb.Booster
    return model.predict(X, num_iteration=model.best_iteration)

thr = float(best_lgb["threshold"])

score_te = predict_score_lgb(best_lgb["model"], X_te)
y_pred = (score_te >= thr).astype(int)

print("best precision under recall constraint")
print(
    "precision:", float(best_lgb["precision"]),
    "recall:", float(best_lgb["recall"]),
    "thr:", thr,
    "best_iteration:", int(best_lgb["model"].best_iteration or 0),
)
print(classification_report(y_te, y_pred, digits=4))

best precision under recall constraint
precision: 0.9950704225352113 recall: 0.7036852589641435 thr: 0.9453966157834216 best_iteration: 1014
              precision    recall  f1-score   support

           0     0.9527    0.9994    0.9755     11981
           1     0.9951    0.7037    0.8244      2008

    accuracy                         0.9570     13989
   macro avg     0.9739    0.8516    0.8999     13989
weighted avg     0.9587    0.9570    0.9538     13989

